# Part 2 - 02 Transfer learning: frozen MobileNetV2 base and new classification head

## Imports and configuration

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from neurovision.data import CLASSES
from neurovision.preprocessing import prepare_data, make_dataset, encoder
from neurovision.transfer import IMAGE_SIZE, build_transfer_model, get_base, count_parameters

RAW_DIR = "../data/raw"
WEIGHTS = "imagenet"
EPOCHS_HEAD = 1

## Load the data (same cleaning and split as Part 1) and compute class weights on train only

In [ ]:
data = prepare_data(RAW_DIR, size=IMAGE_SIZE, seed=42)
X_train, y_train, p_train = data["train"]
X_val, y_val, p_val = data["val"]
X_test, y_test, p_test = data["test"]

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)

weights = compute_class_weight(
    "balanced", classes=np.arange(len(CLASSES)), y=encoder.transform(y_train)
)
class_weight = dict(enumerate(weights))
print(X_train.shape, X_val.shape, class_weight)

## Build the model: frozen MobileNetV2 base, global pooling, dropout, 4-neuron softmax head

In [ ]:
model = build_transfer_model(weights=WEIGHTS)
model.summary()

trainable, non_trainable = count_parameters(model)
print(f"Trainable (head): {trainable:,}")
print(f"Non-trainable (frozen base): {non_trainable:,}")
print("Base trainable:", get_base(model).trainable)

## Callbacks: save the best model, stop early, resume after interruption

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "../models/transfer_best.keras", monitor="val_accuracy", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.BackupAndRestore(backup_dir="../models/backup_transfer"),
]

## Train the new head only and measure the training time

In [ ]:
start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=callbacks,
)
head_minutes = (time.time() - start) / 60
print(f"Head training time: {head_minutes:.1f} min for {len(history.history['loss'])} epoch(s)")

## Learning curves: accuracy and loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.savefig("../reports/transfer_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Validation score before fine-tuning (the test set stays untouched)

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation loss: {val_loss:.4f} | Validation accuracy: {val_acc:.4f}")